In [12]:
import pandas as pd
from sklearn.datasets import load_svmlight_file
import glob
import os

# Path to dataset folder
folder_path = "/Users/kshitijnavale/Desktop/sensor data/Dataset"

# Mapping from label → gas name
gas_mapping = {
    1: "Ethanol",
    2: "Ethylene",
    3: "Ammonia",
    4: "Acetaldehyde",
    5: "Acetone",
    6: "Toluene"
}

all_dfs = []

# Load all .dat files
for file in glob.glob(os.path.join(folder_path, "*.dat")):
    print(f"Processing {file} ...")
    X, y = load_svmlight_file(file, n_features=128)

    # Convert to DataFrame
    df = pd.DataFrame(X.toarray(), columns=[f"f{i}" for i in range(1, 129)])
    df["label"] = y.astype(int)
    df["gas"] = df["label"].map(gas_mapping)

    all_dfs.append(df)

# Combine all batches
final_df = pd.concat(all_dfs, ignore_index=True)

# Save to CSV
output_file = os.path.join(folder_path, "combined_sensor_data_clean.csv")
final_df.to_csv(output_file, index=False)

print("✅ Saved:", output_file)
print("Shape:", final_df.shape)
print("\nLabel distribution:")
print(final_df["gas"].value_counts())


Processing /Users/kshitijnavale/Desktop/sensor data/Dataset/batch8.dat ...
Processing /Users/kshitijnavale/Desktop/sensor data/Dataset/batch9.dat ...
Processing /Users/kshitijnavale/Desktop/sensor data/Dataset/batch4.dat ...
Processing /Users/kshitijnavale/Desktop/sensor data/Dataset/batch5.dat ...
Processing /Users/kshitijnavale/Desktop/sensor data/Dataset/batch7.dat ...
Processing /Users/kshitijnavale/Desktop/sensor data/Dataset/batch6.dat ...
Processing /Users/kshitijnavale/Desktop/sensor data/Dataset/batch2.dat ...
Processing /Users/kshitijnavale/Desktop/sensor data/Dataset/batch3.dat ...
Processing /Users/kshitijnavale/Desktop/sensor data/Dataset/batch1.dat ...
Processing /Users/kshitijnavale/Desktop/sensor data/Dataset/batch10.dat ...
✅ Saved: /Users/kshitijnavale/Desktop/sensor data/Dataset/combined_sensor_data_clean.csv
Shape: (13910, 130)

Label distribution:
gas
Acetone         3009
Ethylene        2926
Ethanol         2565
Acetaldehyde    1936
Toluene         1833
Ammonia   

In [9]:
print(df["label"].value_counts())

Series([], Name: count, dtype: int64)


In [5]:
import pandas as pd
import numpy as np
import os

# Define the directory and file paths
data_dir = "/Users/kshitijnavale/Desktop/sensor data/Dataset"
file_paths = [
    os.path.join(data_dir, f"batch{i}.dat") for i in range(1, 11)
]

# Output CSV path
output_csv = os.path.join(data_dir, "combined_sensor_data.csv")

def read_dat_file(filepath):
    """
    Read a .dat file. Tries common delimiters: space, tab, comma.
    """
    if not os.path.exists(filepath):
        print(f"⚠️ File not found: {filepath}")
        return None
    
    try:
        # Auto-detect separator and read first few rows to check
        sample_df = pd.read_csv(filepath, sep=None, nrows=5, engine='python')
        sep_detected = sample_df.columns[0]  # If no sep detected, it's likely space or tab
        
        # Read full file with detected sep (default to space if unsure)
        if ' ' in open(filepath).readline():
            df = pd.read_csv(filepath, sep='\s+', header=None, engine='python')
        else:
            df = pd.read_csv(filepath, sep=None, header=None, engine='python')
        
        print(f"✅ Loaded {filepath}: {df.shape[0]:,} rows, {df.shape[1]} columns")
        return df
    except Exception as e:
        print(f"❌ Error reading {filepath}: {str(e)}")
        print("💡 Tip: Check if it's binary data or try a different sep (e.g., '\t' for tab).")
        return None

# Read all files
all_dfs = []
for path in file_paths:
    df = read_dat_file(path)
    if df is not None:
        all_dfs.append(df)

if not all_dfs:
    print("❌ No files could be loaded. Check paths and file formats.")
    exit(1)

# Concatenate all dataframes
print(f"\n🔄 Combining {len(all_dfs)} batches...")
combined_df = pd.concat(all_dfs, ignore_index=True)

# Add column names (assuming features f1-fN + label as last column)
num_features = combined_df.shape[1] - 1  # Assume last column is label
feature_cols = [f"f{i+1}" for i in range(num_features)]
all_columns = feature_cols + ['label']
combined_df.columns = all_columns

print(f"📊 Combined dataset shape: {combined_df.shape[0]:,} rows, {combined_df.shape[1]} columns")

# Basic data exploration
print(f"\n📈 Quick Summary:")
print(f"  - Memory usage: {combined_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  - Missing values: {combined_df.isnull().sum().sum()}")
print(f"  - Data types:\n{combined_df.dtypes}")

# Class distribution (if label exists)
if 'label' in combined_df.columns:
    print(f"\n🏷️ Label Distribution:")
    label_counts = combined_df['label'].value_counts().sort_index()
    for label, count in label_counts.items():
        percentage = (count / len(combined_df)) * 100
        print(f"  Label {label}: {count:,} ({percentage:.1f}%)")

# Preview
print(f"\n👀 First 5 rows:\n{combined_df.head()}")

# Save to CSV
try:
    combined_df.to_csv(output_csv, index=False)
    print(f"\n💾 Saved combined data to: {output_csv}")
except Exception as e:
    print(f"❌ Error saving CSV: {str(e)}")

print("\n🎉 Done! You can now use this CSV with your ML pipeline (e.g., quick_training_demo(combined_csv_path)).")

<>:29: SyntaxWarning: invalid escape sequence '\s'
<>:29: SyntaxWarning: invalid escape sequence '\s'
/var/folders/4h/ytmjhp_j7yx1htd0f38lzr_r0000gn/T/ipykernel_85506/2416497196.py:29: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(filepath, sep='\s+', header=None, engine='python')


✅ Loaded /Users/kshitijnavale/Desktop/sensor data/Dataset/batch1.dat: 445 rows, 129 columns
✅ Loaded /Users/kshitijnavale/Desktop/sensor data/Dataset/batch2.dat: 1,244 rows, 129 columns
✅ Loaded /Users/kshitijnavale/Desktop/sensor data/Dataset/batch3.dat: 1,586 rows, 129 columns
✅ Loaded /Users/kshitijnavale/Desktop/sensor data/Dataset/batch4.dat: 161 rows, 129 columns
✅ Loaded /Users/kshitijnavale/Desktop/sensor data/Dataset/batch5.dat: 197 rows, 129 columns
✅ Loaded /Users/kshitijnavale/Desktop/sensor data/Dataset/batch6.dat: 2,300 rows, 129 columns
✅ Loaded /Users/kshitijnavale/Desktop/sensor data/Dataset/batch7.dat: 3,613 rows, 129 columns
✅ Loaded /Users/kshitijnavale/Desktop/sensor data/Dataset/batch8.dat: 294 rows, 129 columns
✅ Loaded /Users/kshitijnavale/Desktop/sensor data/Dataset/batch9.dat: 470 rows, 129 columns
✅ Loaded /Users/kshitijnavale/Desktop/sensor data/Dataset/batch10.dat: 3,600 rows, 129 columns

🔄 Combining 10 batches...
📊 Combined dataset shape: 13,910 rows, 129

Noisy original data

In [ ]:
import pandas as pd
import numpy as np

# Path to the combined real dataset
combined_path = '/Users/kshitijnavale/Desktop/sensor data/Dataset/combined_sensor_data.csv'
output_combined_path = '/Users/kshitijnavale/Desktop/sensor data/Dataset/combined_sensor_data_noisy.csv'

# Load the data
df_combined = pd.read_csv(combined_path)

# Identify feature columns (assuming f1 to f128)
feature_cols = [col for col in df_combined.columns if col.startswith('f')]

# Add Gaussian noise: mean=0, std=0.01 (adjust std as needed)
noise_std = 0.01
noise = np.random.normal(0, noise_std, df_combined[feature_cols].shape)
df_combined[feature_cols] += noise * df_combined[feature_cols].std()

# Optionally apply Gaussian filter (smoothing) to each feature column
# Here, using a simple rolling window approximation for 1D Gaussian filter
window_size = 5  # Adjust for smoothing strength
for col in feature_cols:
    df_combined[col] = df_combined[col].rolling(window=window_size, win_type='gaussian', center=True).mean(std=1)

# Drop any NaN rows introduced by rolling (edges)
df_combined = df_combined.dropna()

# Save the modified dataset
df_combined.to_csv(output_combined_path, index=False)
print(f"Noisy and smoothed combined data saved to: {output_combined_path}")

In [1]:
import pandas as pd
import numpy as np

# Path to the input CSV file
input_path = "/Users/kshitijnavale/Desktop/sensor data/Dataset/synthetic_gas_1M.csv"
output_path = "/Users/kshitijnavale/Desktop/sensor data/Dataset/synthetic_gas_1M_noisier.csv"

# Load the data
print("Loading data...")
df = pd.read_csv(input_path)

# Identify feature columns (assuming they start with 'f')
feature_cols = [col for col in df.columns if col.startswith('f')]

# Parameters for Gaussian noise
noise_std = 0.1  # Standard deviation of the noise, adjust as needed
# You can make it relative to each feature's std: noise_std = df[feature_cols].std() * 0.1

# Add Gaussian noise to each feature column
print("Adding Gaussian noise...")
for col in feature_cols:
    noise = np.random.normal(0, noise_std, size=len(df))
    df[col] += noise

# Save the noisier data to a new CSV
print("Saving noisier data...")
df.to_csv(output_path, index=False)

print(f"Noisier data saved to: {output_path}")

Loading data...
Adding Gaussian noise...
Saving noisier data...
Noisier data saved to: /Users/kshitijnavale/Desktop/sensor data/Dataset/synthetic_gas_1M_noisier.csv


In [23]:
import pandas as pd
import numpy as np

# File path (CSV)
file_path = "/Users/kshitijnavale/Desktop/sensor data/synthetic_gas_1M_realistic_noise.csv"

# Load CSV directly
df = pd.read_csv(file_path)

# Mapping dictionary
gas_mapping = {
    1: "Ethanol",
    2: "Ethylene",
    3: "Ammonia",
    4: "Acetaldehyde",
    5: "Acetone",
    6: "Toluene"
}

# If label column is numeric, map it
if df["label"].dtype in [np.int64, np.float64]:
    df["label"] = df["label"].astype(int)
    df["label"] = df["label"].map(gas_mapping)

print("Shape:", df.shape)
print(df.head())
print("\nLabel distribution:")
print(df["label"].value_counts())


Shape: (1000000, 129)
             f1         f2         f3          f4           f5         f6  \
0 -15423.629627  -9.471007  76.813300  122.997469   285.427683  16.158058   
1  82363.709281 -85.236081 -35.324643  183.523622  1080.992828  26.832616   
2  30704.240910  33.575569 -12.685916  147.152005   -30.456943 -56.562995   
3 -10404.144551  33.575569  96.417812  -18.805283   -30.456943  13.313698   
4  68182.874222 -85.236081 -35.324643  183.523622   -87.035182 -73.114634   

           f7           f8            f9         f10  ...       f120  \
0 -126.322483  -341.988784  -5447.170260  141.689744  ...  14.078777   
1 -134.185865  -911.824503  48946.004104  321.447601  ...  84.833958   
2   36.002975 -1972.388680  52885.024073   22.099883  ...  64.742253   
3 -198.273444   310.516749  32239.122530   22.099883  ...  64.742253   
4 -134.185865  -911.824503  88499.594115  -30.642462  ...  84.833958   

           f121       f122       f123        f124        f125       f126  \
0  103

In [24]:
print(df["label"].value_counts())

label
Ethanol         300000
Ethylene        300000
Acetone         300000
Toluene          80000
Ammonia          10000
Acetaldehyde     10000
Name: count, dtype: int64
